# 04 — Proximal Policy Optimization (PPO)

**Módulo:** `src/llm_rlhf/ppo/`

- `ppo/losses.py` — funciones puras (lee esto primero)
- `ppo/model.py` — wrapper actor + crítico
- `ppo/trainer.py` — bucle externo de rollout/actualización

PPO es el caballo de batalla del RLHF clásico. También es la pieza más delicada del pipeline — la idea general es simple, pero cada detalle importa para la estabilidad.

## El montaje

Tenemos cuatro modelos en juego:

| Rol | Qué es | ¿Se entrena? |
|---|---|---|
| Policy (actor) $\pi_\theta$ | el modelo SFT, genera respuestas | sí |
| Referencia $\pi_{\text{ref}}$ | una copia congelada del modelo SFT | no |
| Crítico $V_\phi$ | predictor escalar de valor por token | sí |
| Modelo de recompensa $r$ | entrenado en el notebook 03 | no (congelado) |

La referencia existe solo para calcular una penalización KL: no queremos que el policy se aleje tanto del inglés sensato que empiece a generar galimatías de reward hacking.

## El objetivo de PPO en una línea

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[ \min\big( \rho_t A_t, \; \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t \big) \Big], \quad \rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$$

Tres piezas, todas cruciales:

1. **Cociente de probabilidades** $\rho_t$ — ¿cuánto más (o menos) probable es el token bajo el nuevo policy que bajo el policy que lo muestreó?
2. **Ventaja** $A_t$ — ¿fue ese token una buena idea? Estimada con GAE a partir de la recompensa por token.
3. **Clipping** — si $\rho_t$ se sale de $[1-\epsilon, 1+\epsilon]$, se ignora la actualización. Esto es lo que hace a PPO "proximal".

Todo lo que hay en `ppo/losses.py` es una de estas tres piezas o las apoya.

### La intuición visual del clipping

La fórmula del *clipped surrogate* es la parte más densa de PPO. Vale la pena verla. La siguiente celda traza el surrogate como función de $\rho$, separando los dos casos:

- **A > 0** (la acción fue buena): el surrogate sube con $\rho$ pero se **aplana arriba** una vez que $\rho > 1+\epsilon$. Esto previene que un gradiente positivo grande nos haga saltar a una política muy distinta.
- **A < 0** (la acción fue mala): el surrogate baja con $\rho$ pero se **aplana abajo** cuando $\rho < 1-\epsilon$. La banda verde es la *región de confianza* — el área donde aún confiamos en la aproximación lineal del gradiente.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# The iconic PPO plot: clipped surrogate as a function of probability ratio rho.
rho = np.linspace(0.0, 2.0, 400)
eps = 0.2

# Two scenarios: A > 0 (good action) and A < 0 (bad action).
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, A, title in zip(axes, [+1.0, -1.0], ['A > 0 (encourage)', 'A < 0 (discourage)']):
    unclipped = rho * A
    clipped   = np.clip(rho, 1 - eps, 1 + eps) * A
    surrogate = np.minimum(unclipped, clipped)
    ax.plot(rho, unclipped, label='rho * A',           linestyle='--', alpha=0.6)
    ax.plot(rho, clipped,   label='clip(rho) * A',    linestyle=':',  alpha=0.6)
    ax.plot(rho, surrogate, label='min(...)  ← PPO',  linewidth=2.5,  color='black')
    ax.axvspan(1 - eps, 1 + eps, color='green', alpha=0.08, label='trust region')
    ax.axvline(1.0, color='gray', linestyle='-', alpha=0.4)
    ax.set_xlabel('rho = pi_new / pi_old')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='lower right' if A > 0 else 'upper right')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('surrogate (gets *maximized*)')
plt.tight_layout()
plt.show()

## Etapa 1 — construyendo la recompensa por token

El modelo de recompensa solo da un escalar por la respuesta entera. PPO necesita una señal por token. Así que construimos:

$$r_t = -\beta \cdot \widehat{\text{KL}}_t \quad \text{para } t < T-1, \qquad r_{T-1} = -\beta \cdot \widehat{\text{KL}}_{T-1} + r_{\text{RM}}$$

La penalización KL por token empuja al policy hacia la referencia en todas partes; la recompensa escalar aparece solo en el último token de la respuesta. No podemos calcular la KL exactamente (no guardamos la distribución completa), así que usamos el estimador $k_3$ de Schulman:

$$\widehat{\text{KL}}_t = e^{\log\rho_t} - 1 - \log\rho_t$$

Siempre es no-negativo y tiene menor varianza que el estimador obvio $\log \rho$.

### ¿Por qué $k_3$ y no simplemente $|\log \rho|$?

Los tres estimadores tienen la misma esperanza (KL verdadera) cuando $\pi$ está cerca de $\pi_{\text{ref}}$. La diferencia está en la *forma local*:

- $k_3$ es **asimétrico** — penaliza más estar arriba (policy más concentrado que la referencia) que estar abajo. Esto encaja con cómo el reward hacking suele manifestarse.
- $|\log \rho|$ es simétrico pero **no diferenciable** en $\rho = 1$.
- $0.5 (\log \rho)^2$ es suave pero crece demasiado lento cerca de cero, dando gradientes pequeños justo donde más los necesitas.

La siguiente celda los superpone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# The three KL estimators from ppo/losses.py, as functions of log_ratio = log(pi/pi_ref).
log_ratio = np.linspace(-1.5, 1.5, 200)

k3   = np.exp(log_ratio) - 1 - log_ratio
absv = np.abs(log_ratio)
mse  = 0.5 * log_ratio ** 2

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(log_ratio, k3,   label='k_3 (Schulman): exp(log_r) - 1 - log_r', linewidth=2)
ax.plot(log_ratio, absv, label='|log_r|', linestyle='--')
ax.plot(log_ratio, mse,  label='0.5 (log_r)^2', linestyle=':')
ax.axvline(0, color='gray', alpha=0.4)
ax.set_xlabel('log_ratio = log(pi / pi_ref)')
ax.set_ylabel('KL estimate per token')
ax.set_title('KL estimators — k_3 is asymmetric and always non-negative')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from llm_rlhf.ppo.losses import compute_rewards_with_kl_penalty
import torch

# Toy example: two responses, three tokens each
policy_lp = torch.tensor([[-0.5, -0.7, -0.4], [-0.3, -0.2, -0.6]])
ref_lp    = torch.tensor([[-0.6, -0.4, -0.5], [-0.4, -0.5, -0.5]])
mask      = torch.ones(2, 3)
scalar_rm = torch.tensor([1.2, -0.4])  # reward-model scores

rewards, kl = compute_rewards_with_kl_penalty(scalar_rm, policy_lp, ref_lp, mask, kl_coef=0.1)
print('per-token reward:'); print(rewards)
print('per-token KL    :'); print(kl)

## Etapa 2 — ventajas vía GAE

Dada la recompensa por token, GAE estima cuánto *mejor* fue la acción elegida frente al baseline de valor:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \qquad A_t = \sum_{l=0}^{T-1-t} (\gamma \lambda)^l \, \delta_{t+l}$$

Iteramos hacia atrás por la secuencia (cada $A_t$ reutiliza la suma corriente del siguiente paso) y sumamos $V_t$ de vuelta para obtener $R_t$, el target de regresión para la cabeza de valor.

El truco: **blanqueamos** las ventajas (z-score) sobre el batch antes de usarlas. La pérdida del policy es sensible a la *escala* de $A_t$, y el blanqueo estabiliza mucho el entrenamiento.

In [ ]:
from llm_rlhf.ppo.losses import compute_gae

values = torch.tensor([[0.1, 0.0, 0.2], [-0.2, 0.1, 0.0]])
adv, ret = compute_gae(rewards, values, mask, gamma=1.0, lam=0.95, whiten=False)
print('advantages:'); print(adv)
print('returns   :'); print(ret)

### GAE, paso a paso

Construyamos un ejemplo claro: 8 tokens, recompensa cero excepto un valor grande al final (la recompensa del RM), y un baseline de valor que aumenta con la posición.

Mira cómo la ventaja se propaga hacia atrás: el *bonus* del último token "fluye" hacia los pasos anteriores con peso decreciente $(\gamma\lambda)^l$, dándoles crédito proporcional.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize GAE on a slightly longer toy trajectory so the shape is more readable.
T = 8
r_toy = torch.tensor([[0.0, 0.0, 0.0, 0.0, 0.0, -0.1, 0.0, 1.5]])      # reward only at the end
v_toy = torch.tensor([[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]])       # rising value baseline
m_toy = torch.ones(1, T)
adv_toy, ret_toy = compute_gae(r_toy, v_toy, m_toy, gamma=1.0, lam=0.95, whiten=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
xs = np.arange(T)
axes[0].bar(xs, r_toy[0].numpy(), color='#4a90d9'); axes[0].set_title('per-token reward r_t')
axes[1].bar(xs, v_toy[0].numpy(), color='#cccccc'); axes[1].set_title('value baseline V(s_t)')
axes[2].bar(xs, adv_toy[0].numpy(), color='#ff7f50'); axes[2].set_title('advantage A_t (GAE)')
for ax in axes:
    ax.set_xlabel('token position')
    ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('returns R_t:', ret_toy[0].numpy())

## Etapa 3 — la pérdida con clipping

Ahora juntamos todo. La pérdida del policy es el *clipped surrogate*. La pérdida del valor es MSE sobre los retornos GAE, también con clipping (un truco fuera del paper original que ayuda en la práctica). Un bonus de entropía promueve la exploración.

$$L = L^{\text{CLIP}} + c_v \, L^{\text{value}} - c_e \, H[\pi_\theta]$$

Cada una de estas es un one-liner en `ppo/losses.py`. El trabajo del trainer es ensamblarlas, no calcularlas.

## El bucle externo

```
for iteration in range(N):
    1. ROLLOUT    sample responses, record old log-probs, old values, RM score
    2. COMPUTE    KL-penalty rewards -> GAE advantages and returns
    3. UPDATE     for ppo_epochs:
                      recompute log-probs/values with current policy
                      apply clipped policy loss + value loss + entropy bonus
```

El bucle interno `ppo_epochs` es lo que hace a PPO eficiente en datos — sacamos varias actualizaciones de gradiente de un mismo rollout. El clipping es lo que evita que esas actualizaciones se alejen demasiado.

In [ ]:
from llm_rlhf.ppo import PPOTrainer, PPOConfig
from llm_rlhf.config import load_toml

ppo_cfg = PPOConfig(**load_toml('../configs/ppo.toml'))
print(ppo_cfg)

## Qué puede salir mal

Modos de fallo de PPO que vale la pena conocer:

- **Reward hacking** — el policy encuentra galimatías con recompensa alta. Mitigado por la penalización KL.
- **Colapso del valor** — la pérdida del crítico explota; las ventajas se vuelven ruido. Mitigado por value clipping + blanqueo.
- **Colapso de modo** — el policy se vuelve demasiado determinista. Mitigado por el bonus de entropía.
- **Drift distribucional** — el policy se aleja tanto de la referencia que el modelo de recompensa queda fuera de distribución. Mitigado por la penalización KL *y* por parar cuando la KL crece.

Los siguientes dos algoritmos (DPO, GRPO) son respuestas a estas dificultades — DPO elimina el bucle de RL por completo, GRPO elimina la función de valor.

## Ejercicio: sensibilidad a `clip_epsilon` y `kl_coef`

Las dos perillas más importantes de PPO:

- `clip_epsilon` controla el ancho de la región de confianza. Demasiado pequeño → casi no se aprende; demasiado grande → updates inestables.
- `kl_coef` controla cuánto puede alejarse el policy de la referencia. Demasiado pequeño → reward hacking; demasiado grande → el policy se queda igual al SFT.

La siguiente celda recorre varios valores sobre los tensores de juguete que ya tenemos. Antes de correrla, predice:

- Si `clip_epsilon` aumenta, ¿la fracción de tokens clipeados debería subir o bajar?
- Si `kl_coef` aumenta, ¿la recompensa neta en el último token debería subir o bajar?

In [ ]:
# Exercise: sensitivity to clip_epsilon and kl_coef
# Reusing the toy rewards/values from earlier, sweep these hyperparameters
# and observe how the policy loss changes for the same data.
from llm_rlhf.ppo.losses import compute_policy_loss

# Pretend the policy moved from old logprobs to slightly-shifted new logprobs.
old_lp = policy_lp.clone()
new_lp = policy_lp + torch.tensor([[+0.30, -0.05, +0.10],
                                   [-0.20, +0.40, +0.05]])

print(f'{"clip_epsilon":>14}  {"policy_loss":>12}  {"clip_frac":>10}  {"approx_kl":>10}')
for eps in [0.05, 0.1, 0.2, 0.4]:
    loss, stats = compute_policy_loss(old_lp, new_lp, adv, mask, clip_epsilon=eps)
    print(f'{eps:>14.2f}  {loss.item():>12.4f}  {stats["clip_frac"]:>10.3f}  {stats["approx_kl"]:>10.4f}')

# Try the same sweep with kl_coef inside compute_rewards_with_kl_penalty:
print()
print(f'{"kl_coef":>14}  {"reward[end]":>12}')
for k in [0.0, 0.05, 0.2, 0.5]:
    r, _ = compute_rewards_with_kl_penalty(scalar_rm, policy_lp, ref_lp, mask, kl_coef=k)
    print(f'{k:>14.2f}  {r[0, -1].item():>12.4f}')

## Siguiente: `05_dpo.ipynb`

DPO esquiva PPO por completo. Misma señal de preferencia, sin rollouts, sin crítico, sin clipping. Veremos cómo.